# C3-A8 · Hands-on: a conditional diffusion model for PV disaggregation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcjene/ai4gd-c3-diffusion/blob/main/AI4GD_C3-A8.ipynb)

**AI4GreenDeal · Course C3, Conditional diffusion modeling for energy disaggregation · about 120 min**

In C3-A5 you saw how a conditional diffusion model (cDDPM) learns and generates the PV behind a smart meter, in C3-A6
what is inside its noise predictor, and in C3-A7 how to turn its samples into prediction intervals. Now you write the key
pieces yourself, train the model and evaluate it.

| Section | Topic | min |
|---|---|---|
| 0 | Setup | 5 |
| 1 | Data and conditions | 10 |
| 2 | Forward process | 12 |
| 3a | Inside the U-Net | 15 |
| 3 | Training | 15 |
| 4 | Sampling | 18 |
| 5 | From samples to estimates | 20 |
| 6 | What does the model listen to? | 20 |
| 7 | Wrap-up | 5 |

**How the exercises work.** Replace every `____` with your code and run the cell. `ex1.check(...)` tells you whether
it is right, and if not, what is probably wrong. Stuck? Uncomment `ex1.hint()` or `ex1.solution()`. If a check
fails, the notebook still goes on: the next cells use the reference solution and say so. Questions are answered with
`q1.check("B")`. Each section starts with a **catch-up cell**: after a runtime restart, run it and carry on.

> The numbers in this notebook are **illustrations, not results of the paper**: the course models use a different
> noise schedule (β up to 0.2 instead of 0.02), and we evaluate on a few days only.

**Runtime.** Choose a GPU: *Runtime → Change runtime type → T4 GPU*. Without a GPU the notebook still runs: training
is replaced by the course checkpoint, each day gets 20 samples instead of 50, Section 6 uses 3 days and the optional Ex8c
is skipped.

**Notation** (as in C3-A5 to C3-A7): $x_0$ one day of PV, $T = 48$ half-hours; $x_k$ the noisy profile at
diffusion step $k = 1 \dots K$, $K = 100$; $c$ the conditions; $\varepsilon_\theta(x_k, k, c)$ the noise predicted by
the network; $\bar\alpha_k$ the signal level; $pv_i$, $\hat{pv}_i$ true and estimated PV at half-hour $i$.

<div style="height:8px;background:linear-gradient(90deg,#1f5f99,#3a8fd6);border-radius:4px"></div>

## <span style="color:#2f7fc8">Section 0</span> · Setup <small>(≈ 5 min)</small>

Install the course package, check the GPU and download the data.

In [ ]:
# Install the course package (on Colab), check the GPU, download the data (14 MB)
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "git+https://github.com/marcjene/ai4gd-c3-diffusion"], check=True)

import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import ai4gd_c3 as c3
from ai4gd_c3 import data, model, plots, reference
from ai4gd_c3.exercises import *   # ____, ex1 ... ex9, q1 ... q8, progress, catch_up
from IPython.display import Markdown

GPU = c3.DEVICE != "cpu"
print("device:", torch.cuda.get_device_name() if c3.DEVICE == "cuda" else c3.DEVICE)
if not GPU:
    print("No GPU: Section 3 loads the course checkpoint instead of training, 20 samples per day instead of 50,\n"
          "Section 6 uses 3 days, Ex8c is skipped.")
D = data.load()
print(f"data: {len(D['customers'])} Ausgrid customers ({D['is_train'].sum()} for training), {len(D['pv'])} days")

<div style="height:8px;background:linear-gradient(90deg,#1f5f99,#3a8fd6);border-radius:4px"></div>

## <span style="color:#2f7fc8">Section 1</span> · Data and conditions <small>(≈ 10 min)</small>

The model generates one day of PV, $x_0$, guided by three conditions $c$: the irradiance $\mathrm{GHI}_d$ and the
net consumption of the same day, $\mathrm{NC}_d$, and of the previous day, $\mathrm{NC}_{d-1}$. The data are the
Ausgrid Solar Home dataset (Sydney, July 2012 to June 2013, 30 min) with GHI from Open-Meteo.

As in C3-A5, PV is divided by the installed capacity $C$ (kWp), so that $x_0 = PV / C$; $\mathrm{NC}_d$ and
$\mathrm{NC}_{d-1}$ are divided by the household's maximum export (kW); GHI is in kW/m².

We follow customer 134 from C3-A5 to C3-A7.

In [ ]:
# Catch-up, Section 1
S = catch_up(1)
d = S.day   # customer 134, clear day 2013-01-04
print(f"customer {d['customer']}, {d['date']}: C = {d['capacity_kwp']:.2f} kWp, max. export {d['max_export_kw']:.2f} kW")

C3-A7 rounded this household to 1.5 kWp; here we use the measured value, so kW values differ by about 1 %.

📖 **Read** Section 2.4 of [1], *Conditioning strategy* (about 3 min). **Find out:** why the paper divides PV and NC
by a physical value instead of using min–max or z-score scaling. You need it for Ex1.

> **Note.** [1] divides NC by the installed PV capacity; the course models divide it by the household's maximum export
> (as C3-A5 explains). PV is divided by the capacity in both.

**Ex1.** Convert the day from kW to the model's per-unit values.

In [ ]:
def normalise(pv_kw, nc_kw, capacity_kwp, max_export_kw):
    '''Convert PV and net consumption from kW to per-unit.'''
    pv_pu = ____
    nc_pu = ____
    return pv_pu, nc_pu

ex1.check(normalise)
# ex1.hint()
# ex1.solution()

In [ ]:
normalise = ex1.use(normalise)
pv_pu, nc_pu = normalise(d["pv_kw"], d["nc_kw"], d["capacity_kwp"], d["max_export_kw"])
print(f"peak PV {d['pv_kw'].max():.2f} kW = {pv_pu.max():.2f} pu;  lowest NC {d['nc_kw'].min():.2f} kW = {nc_pu.min():.2f} pu")
plots.day(d)

**Look at your data.** Around noon NC drops below zero: the household exports, and that is the PV imprint on the
meter. Over the whole dataset, two values look odd at first:

In [ ]:
print(f"largest PV value: {D['pv'].max():.2f} pu of installed capacity")
print(f"largest NC value: {D['nc'].max():.1f} pu of maximum export")

PV above 1 pu happens when a system produces more than its listed capacity (nameplate understating the array,
metadata errors or metering effects). NC values far above 1 pu need more thought.

<div style="background:rgba(232,160,32,0.13);border-left:5px solid #e8a020;padding:10px 16px;border-radius:4px">
<b>q1.</b> A household has a 3 kWp PV system, but its meter never exported more than 0.3 kW in the whole year. What does this most likely mean for the model?
<ul style="margin:8px 0 2px 0"><li><b>A</b>&nbsp; Its inverter is probably export-limited, so PV generation itself is capped near 0.3 kW and x<sub>0</sub> rarely exceeds 0.1 pu.</li><li><b>B</b>&nbsp; The array is probably shaded or degraded; dividing PV by the capacity removes this effect, so the conditions are not affected.</li><li><b>C</b>&nbsp; Most of its PV is consumed on site, so NC<sub>d</sub> carries a weak PV imprint and large pu values; the model has to lean more on GHI and NC<sub>d−1</sub>.</li><li><b>D</b>&nbsp; For this household NC<sub>d</sub> stays within [−1, 1] pu, which makes it a better-scaled input than for households that export a lot.</li></ul>
</div>

In [ ]:
q1.check(____)   # A, B, C or D

<div style="height:8px;background:linear-gradient(90deg,#1f5f99,#3a8fd6);border-radius:4px"></div>

## <span style="color:#2f7fc8">Section 2</span> · Forward process <small>(≈ 12 min)</small>

To learn to remove noise, the model needs noisy examples. We mix the real PV with Gaussian noise, following a linear
schedule from $\beta_1 = 10^{-4}$ to $\beta_K = 0.2$ with $K = 100$, so that $x_K$ is pure noise:

$$\alpha_k = 1 - \beta_k, \qquad \bar\alpha_k = \prod_{j=1}^{k} \alpha_j, \qquad
x_k = \sqrt{\bar\alpha_k}\, x_0 + \sqrt{1 - \bar\alpha_k}\, \varepsilon, \quad \varepsilon \sim N(0, I).$$

**Index convention.** The equations count $k$ from 1, Python counts from 0. We store the schedule with $K + 1$
entries and put $\beta_0 = 0$, $\alpha_0 = \bar\alpha_0 = 1$ at index 0. Then `abar[k]` is exactly $\bar\alpha_k$ of
C3-A5 for $k = 1 \dots K$, and step 0 is $x_0$ itself. (Inside, the network still receives $k - 1$, because the course
checkpoints were trained that way; you never have to think about it.)

In [ ]:
# Catch-up, Section 2
S = catch_up(1)
d = S.day
x0 = d["pv_pu"]

**Ex2.** Build the schedule.

In [ ]:
def linear_schedule(K, beta_1=1e-4, beta_K=0.2):
    '''Linear noise schedule as tensors of length K + 1 (index 0: beta_0 = 0, abar_0 = 1).'''
    beta = ____
    alpha = ____
    abar = ____
    return beta, alpha, abar

ex2.check(linear_schedule)
# ex2.hint()
# ex2.solution()

In [ ]:
linear_schedule = ex2.use(linear_schedule)
beta, alpha, abar = linear_schedule(model.K)
print(f"abar_0 = {abar[0]:.3f}, abar_1 = {abar[1]:.5f}, abar_K = {abar[model.K]:.1e}")

**Ex3.** Jump straight to step $k$ with the closed form. `x0` and `eps` have shape `[B, T]`; `k` holds one
step per profile, shape `[B]`, integers $1 \dots K$.

In [ ]:
def q_sample(x0, k, eps, abar):
    '''Noisy profiles x_k from the closed-form forward process.'''
    a = ____
    return ____

ex3.check(q_sample)
# ex3.hint()
# ex3.solution()

In [ ]:
q_sample = ex3.use(q_sample)
torch.manual_seed(0)
eps = torch.randn(1, 48)
plots.forward(x0, abar, [0, 10, 27, 50, 100], lambda k: q_sample(x0[None], torch.tensor([k]), eps, abar)[0])

In [ ]:
print(f"k = 10: signal share {abar[10].sqrt():.2f}, noise share {(1 - abar[10]).sqrt():.2f}")
print(f"standard deviation of this PV day: {x0.std():.2f} pu (the noise has standard deviation 1)")
print("signal and noise shares cross at k =", int((abar < 0.5).nonzero()[0]))

<div style="background:rgba(232,160,32,0.13);border-left:5px solid #e8a020;padding:10px 16px;border-radius:4px">
<b>q2.</b> At k = 10 the signal share is still about 0.95, yet in the plot the daily PV shape is already hard to see. Why?
<ul style="margin:8px 0 2px 0"><li><b>A</b>&nbsp; The PV values are small next to unit-variance noise: a noise share of 0.3 is already as large as the PV's own variation over the day.</li><li><b>B</b>&nbsp; The linear schedule adds most of its noise in the first steps, so the first ten steps destroy the most signal.</li><li><b>C</b>&nbsp; The noise is drawn independently for each half-hour, which breaks the daily shape faster than it lowers the PV level.</li><li><b>D</b>&nbsp; √ᾱ<sub>k</sub> already shrinks the PV profile to a fraction of its size, so there is little left to see.</li></ul>
</div>

In [ ]:
q2.check(____)   # A, B, C or D

<div style="height:8px;background:linear-gradient(90deg,#1f5f99,#3a8fd6);border-radius:4px"></div>

## <span style="color:#2f7fc8">Section 3a</span> · Inside the U-Net <small>(≈ 15 min)</small>

The noise predictor $\varepsilon_\theta(x_k, k, c)$ is the 1D U-Net of Fig. 3 in [1] (C3-A6, screen 1).

📖 **Read** Section 2.3 of [1], *Model architecture* (about 5 min). **Find out:** where the conditions enter the network,
and how the diffusion step $k$ reaches the convolutional blocks. You need both for Ex4a and Ex4b.

| Block | What it does |
|---|---|
| `input` | stack $x_k$ and the 3 conditions as channels |
| `init_conv` | Conv1d, kernel 3, to $B_{ch} = 128$ channels |
| `encoder_1` | modulated convolution, doubles the channels (copy → skip 1) |
| `encoder_2` | max pooling ÷2, modulated convolution, doubles the channels (copy → skip 2) |
| `bottleneck` | max pooling ÷2, two modulated convolutions (to 1024 channels and back) |
| `decoder_2` | transposed convolution ×2, join skip 2, modulated convolution to 256 channels |
| `decoder_1` | transposed convolution ×2, join skip 1, modulated convolution to 128 channels |
| `output` | Conv1d to 1 channel, squeezed: $\varepsilon_\theta$ has the shape of $x_k$ |

FiLM in each modulated convolution, with $(\gamma, \delta)$ computed from the step embedding of $k$:
$h \leftarrow \mathrm{ReLU}\big(h \cdot (1 + \gamma) + \delta\big)$.

In the course model, the conditions are stacked with $x_k$ as input channels, and the step $k$ enters every
modulated convolution through FiLM.

**Ex4a.** Predict the output shape of every block for a batch of $B = 16$ profiles.

In [ ]:
shapes = {
    "input":      (16, 4, 48),   # given
    "init_conv":  ____,
    "encoder_1":  ____,
    "encoder_2":  ____,
    "bottleneck": ____,
    "decoder_2":  ____,
    "decoder_1":  ____,
    "output":     ____,
}
ex4a.check(shapes)
# ex4a.hint()

**Ex4b.** Complete the U-Net. **Blank 1** is FiLM: `self.film(t_emb)` gives a scale and a shift per channel, each
`[B, out_ch]`, applied as $h \cdot (1 + \text{scale}) + \text{shift}$ along the whole length. **Blank 2** joins the
decoder input with its skip connection. Keep all layer names: the check loads the course weights (`pv_all.pt`) into
your class and compares the outputs.

In [ ]:
from ai4gd_c3.model import TimestepEmbedding


class ModulatedConv1D(nn.Module):
    '''Conv1d whose output is scaled and shifted according to the step k (FiLM).'''
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1)
        self.film = nn.Linear(time_emb_dim, out_ch * 2)

    def forward(self, x, t_emb):
        h = self.conv(x)
        # blank 1: FiLM
        scale, shift = ____
        h = ____

        return F.relu(h)


class UNet1D(nn.Module):
    def __init__(self, in_ch, base_channels, time_embed_dim, diffusion_steps, depth=2):
        super().__init__()
        self.time_embedding = TimestepEmbedding(time_embed_dim, base_channels)
        self.init_conv = nn.Conv1d(in_ch, base_channels, kernel_size=3, padding=1)
        self.enc_blocks = nn.ModuleList()
        self.pool = nn.MaxPool1d(kernel_size=2)
        ch, skip_ch = base_channels, []
        for _ in range(depth):
            self.enc_blocks.append(ModulatedConv1D(ch, ch * 2, base_channels))
            skip_ch.append(ch)
            ch *= 2
        self.bot1 = ModulatedConv1D(ch, ch * 2, base_channels)
        self.bot2 = ModulatedConv1D(ch * 2, ch, base_channels)
        self.up_blocks, self.dec_blocks = nn.ModuleList(), nn.ModuleList()
        for s in reversed(skip_ch):
            self.up_blocks.append(nn.ConvTranspose1d(ch, s, kernel_size=2, stride=2))
            self.dec_blocks.append(ModulatedConv1D(ch + s, s, base_channels))
            ch = s
        self.final_conv = nn.Conv1d(ch, 1, kernel_size=3, padding=1)

    def forward(self, x, k, c):
        '''x_k [B, T], steps k [B], conditions c [B, 3, T] -> eps_theta [B, T].'''
        t_emb = self.time_embedding(k - 1)                          # the checkpoints use 0-based steps
        x = self.init_conv(torch.cat([x.unsqueeze(1), c], dim=1))   # input -> initial convolution
        skips = []
        for enc in self.enc_blocks:                                 # encoder levels 1, 2
            x = F.relu(enc(x, t_emb))
            skips.append(x)
            x = self.pool(x)
        x = F.relu(self.bot1(x, t_emb))                             # bottleneck
        x = F.relu(self.bot2(x, t_emb))
        for up, dec in zip(self.up_blocks, self.dec_blocks):        # decoder levels 2, 1
            skip = skips.pop()
            x = up(x)
            x = ____                                                # blank 2: skip connection

            x = F.relu(dec(x, t_emb))
        return self.final_conv(x).squeeze(1)                        # output


ex4b.check(UNet1D)
# ex4b.hint()
# ex4b.solution()

In [ ]:
UNet = ex4b.use(UNet1D)
net = model.new_model("all", cls=UNet)
print(f"{sum(p.numel() for p in net.parameters()):,} parameters")   # 5,421,569 with the configuration of [1]

<div style="background:rgba(232,160,32,0.13);border-left:5px solid #e8a020;padding:10px 16px;border-radius:4px">
<b>q3.</b> One bottleneck value sees 28 input half-hours (about 14 h, C3-A6 screen 2). Which single change would let it see the whole day (48 half-hours)?
<ul style="margin:8px 0 2px 0"><li><b>A</b>&nbsp; two more modulated convolutions in the bottleneck</li><li><b>B</b>&nbsp; doubling the base channels to 256, so that each position can encode more context</li><li><b>C</b>&nbsp; replacing every max pooling by a strided convolution (kernel 2, stride 2)</li><li><b>D</b>&nbsp; a third U-Net level (depth 3)</li></ul>
</div>

In [ ]:
q3.check(____)   # A, B, C or D

<div style="height:8px;background:linear-gradient(90deg,#1f5f99,#3a8fd6);border-radius:4px"></div>

## <span style="color:#2f7fc8">Section 3</span> · Training <small>(≈ 15 min)</small>

One training iteration (Algorithm 1 in [1], C3-A5 screen 4):

1. take a batch of real days $x_0$ with their conditions $c$;
2. draw one step $k$, uniformly from $\{1, \dots, K\}$, and noise $\varepsilon \sim N(0, I)$ per profile;
3. noise the profiles: $x_k = \sqrt{\bar\alpha_k}\, x_0 + \sqrt{1 - \bar\alpha_k}\, \varepsilon$;
4. loss $= \text{mean of } \big(\varepsilon - \varepsilon_\theta(x_k, k, c)\big)^2$; one Adam step.

The course model repeats this 2000 times, with batch 256 and learning rate $10^{-3}$, over the 107 training households.

📖 **Read** Section 2.2 of [1], from the start to the end of Algorithm 1 (about 5 min). **Find out:** how the step $k$ is
chosen at each training iteration, and what the network is trained to predict. You need both for Ex4c.

In [ ]:
# Catch-up, Section 3
S = catch_up(2)
beta, alpha, abar = S.schedule
q_sample = ex3.use(globals().get("q_sample", reference.q_sample))
UNet = ex4b.use(globals().get("UNet1D", model.UNet1D))

**Ex4c.** Write the loss of one training iteration (steps 2–4). Create `k` on the same device as `x0`.

In [ ]:
def diffusion_loss(model, x0, c, abar, K):
    '''Loss of one training iteration (Algorithm 1).'''
    k = ____
    eps = ____
    x_k = ____
    return ____

ex4c.check(diffusion_loss)
# ex4c.hint()
# ex4c.solution()

Now train the model with all three conditions. With a T4 GPU this takes a few minutes. `TRAIN_IN_NOTEBOOK = False`
loads the course checkpoint `pv_all.pt` instead: the same configuration, already trained.

In [ ]:
TRAIN_IN_NOTEBOOK = True

diffusion_loss = ex4c.use(diffusion_loss)
TRAINED = TRAIN_IN_NOTEBOOK and GPU
if TRAINED:
    net = model.new_model("all", cls=UNet)
    t0 = time.time()
    losses = model.train(net, diffusion_loss, abar)
    print(f"trained in {(time.time() - t0) / 60:.1f} min")
    plots.losses(losses)
else:
    print("↪ loading the course checkpoint pv_all.pt instead of training" + ("" if GPU else " (no GPU)"))
    net = model.load_pretrained("all", cls=UNet)

The loss falls fast in the first few hundred steps and then flattens. It stays noisy because every batch mixes
different days and different steps $k$: predicting the noise is easy at large $k$ and hard at small $k$.

<div style="height:8px;background:linear-gradient(90deg,#1f5f99,#3a8fd6);border-radius:4px"></div>

## <span style="color:#2f7fc8">Section 4</span> · Sampling <small>(≈ 18 min)</small>

Generation starts from pure noise $x_K \sim N(0, I)$ and applies the network once per step, from $k = K$
down to 1 (Algorithm 2 in [1], C3-A5 screen 5):

$$x_{k-1} = \frac{1}{\sqrt{\alpha_k}} \left( x_k - \frac{\beta_k}{\sqrt{1 - \bar\alpha_k}}\,
\varepsilon_\theta(x_k, k, c) \right) + \sigma_k z, \qquad \sigma_k = \sqrt{\beta_k},\quad z \sim N(0, I),$$

with no fresh noise at the last step ($z = 0$ for $k = 1$).

📖 **Read** Section 2.2 of [1], from "Once the model is trained" to the end of Algorithm 2 (about 5 min). **Find out:** how
the noise added at each reverse step is scaled, at which step it is left out, and why. You need it for Ex5.

In [ ]:
# Catch-up, Section 4
S = catch_up(4)
beta, alpha, abar = schedule = S.schedule
net = globals().get("net", S.model)
TRAINED = globals().get("TRAINED", False) and net is not S.model
course = S.model
d = S.day

**Ex5.** One reverse step. `k` is a Python integer here; `z` is given so that the check is reproducible.

In [ ]:
def reverse_step(x_k, eps_hat, k, beta, alpha, abar, z):
    '''One reverse step x_k -> x_(k-1) (Algorithm 2).'''
    mean = ____
    return ____

ex5.check(reverse_step)
# ex5.hint()
# ex5.solution()

Generate 50 samples (20 without a GPU) for customer 134 on the clear day, with your model and with the course model (same initial
noise). They should look alike, but not identical: training is random. (If you did not train in Section 3, your model *is*
the course model, and you see one panel.)

In [ ]:
reverse_step = ex5.use(reverse_step)
mine = model.sample(net, d["c"], schedule, reverse_step)
panels = [(mine, d, "your model" if TRAINED else "course model pv_all.pt")]
if TRAINED:   # same initial noise, course checkpoint
    panels.append((model.sample(course, d["c"], schedule, reverse_step), d, "course model pv_all.pt"))
plots.compare(1, len(panels), [(s, dd, f"{title}, raw samples") for s, dd, title in panels])

**Ex5b. Watch the reverse process.** At every step the network's noise estimate also gives a guess of the final
profile:

$$\hat{x}_0 = \frac{x_k - \sqrt{1 - \bar\alpha_k}\; \varepsilon_\theta(x_k, k, c)}{\sqrt{\bar\alpha_k}}.$$

Predict first, then write $\hat{x}_0$ and run the cells below.

<div style="background:rgba(232,160,32,0.13);border-left:5px solid #e8a020;padding:10px 16px;border-radius:4px">
<b>q4.</b> <i>Predict first.</i> On the clear day, at which step will x̂<sub>0</sub> first show the daily PV shape, and why?
<ul style="margin:8px 0 2px 0"><li><b>A</b>&nbsp; around k = 27, where the signal share overtakes the noise share</li><li><b>B</b>&nbsp; around k = 80: ε̂ is accurate enough that removing it reveals the shape long before x<sub>k</sub> shows it</li><li><b>C</b>&nbsp; around k = 50: before that, dividing by √ᾱ<sub>k</sub> amplifies the errors in ε̂ too much</li><li><b>D</b>&nbsp; around k = 10, because x̂<sub>0</sub> cannot be better than the x<sub>k</sub> it is computed from</li></ul>
</div>

In [ ]:
q4.check(____)   # A, B, C or D

In [ ]:
def predict_x0(x_k, eps_hat, k, abar):
    '''The network's guess of x_0 at step k.'''
    return ____

ex5b.check(predict_x0)

In [ ]:
predict_x0 = ex5b.use(predict_x0)
ks = [100, 80, 60, 40, 20, 10]
days = {label: data.day(data.CUSTOMER, date) for label, date in data.DAYS_134.items()}
final, kept = model.sample(course, torch.stack([dd["c"] for dd in days.values()]), schedule, reverse_step,
                           record=ks)


def med_corr(profiles, truth):
    '''Median over the samples of the correlation with the true PV.'''
    return torch.stack([torch.corrcoef(torch.stack([p, truth]))[0, 1] for p in profiles]).nanmedian().item()


print(f"median correlation with the true PV over {final.shape[1]} samples")
print("            " + "".join(f"k = {k:<5}" for k in ks + [0]))
for i, (label, dd) in enumerate(days.items()):
    hat = [med_corr(predict_x0(kept[k][0][i], kept[k][1][i], k, abar), dd["pv_pu"]) for k in ks]
    raw = [med_corr(kept[k][0][i], dd["pv_pu"]) for k in ks]
    last = med_corr(final[i], dd["pv_pu"])
    print(f"day {label}  x̂_0  " + "".join(f"{v:<9.2f}" for v in hat + [last]))
    print(f"       x_k  " + "".join(f"{v:<9.2f}" for v in raw + [last]))

b = list(days).index("b")
plots.trajectory(ks + [0], [[kept[k][0][b, 0] for k in ks] + [final[b, 0]],
                            [predict_x0(kept[k][0][b, 0], kept[k][1][b, 0], k, abar) for k in ks] + [final[b, 0]]],
                 days["b"]["pv_pu"])

$\hat{x}_0$ shows the shape long before $x_k$ does: the network guesses the overall profile early, and the later steps
refine the detail. On the overcast day (c) the shape appears much later: there is little PV to recover.

**Ex6. Post-processing.** The raw samples can be negative and are not zero at night. As in C3-A5: clamp at 0, and set
PV to 0 wherever GHI = 0. Return a new tensor rather than changing `samples` in place.

In [ ]:
def postprocess(samples, ghi):
    '''Post-process raw samples [N, T] with the GHI of the day [T].'''
    return ____

ex6.check(postprocess)
# ex6.hint()

In [ ]:
postprocess = ex6.use(postprocess)
plots.compare(1, len(panels), [(postprocess(s, d["ghi"]), dd, title) for s, dd, title in panels])

<div style="height:8px;background:linear-gradient(90deg,#1f5f99,#3a8fd6);border-radius:4px"></div>

## <span style="color:#2f7fc8">Section 5</span> · From samples to estimates <small>(≈ 20 min)</small>

From here on we use the course model `pv_all.pt`, so that the numbers match C3-A7, and work in kW. Keep the one-page
summary from C3-A7 open: *Metrics at a glance* (link printed by the catch-up cell below). Definitions as in C3-A7 and
[1], Section 3.2. $N = 48$ for RMSE; $N_{sun}$ counts the daylight half-hours ($\mathrm{GHI} > 0$), and
$C$ is the installed capacity.

$$\mathrm{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (pv_i - \hat{pv}_i)^2}, \qquad
\mathrm{PICP} = \frac{100}{N_{sun}} \sum_{i=1}^{N_{sun}} c_i, \qquad
\mathrm{PINAW} = \frac{1}{N_{sun}} \sum_{i=1}^{N_{sun}} \frac{\hat{pv}_i^{\,q_u} - \hat{pv}_i^{\,q_\ell}}{C},$$

with $c_i = 1$ if $pv_i$ lies inside the interval, and the pinball loss of quantile $q$, averaged over
$q = 0.1, \dots, 0.9$ and the daylight half-hours:

$$PL_i^q = \begin{cases} (1 - q)\,(\hat{pv}_i^{\,q} - pv_i) & \text{if } pv_i < \hat{pv}_i^{\,q} \\
q\,(pv_i - \hat{pv}_i^{\,q}) & \text{otherwise.} \end{cases}$$

📖 **Read** Section 3.2 of [1], *Evaluation metrics* (about 5 min). **Find out:** over which half-hours the paper computes
PICP, PINAW and the pinball loss, and why. You need it for Ex8 (the notebook uses GHI > 0 for the same purpose).

In [ ]:
# Catch-up, Section 5: course-model samples for days a (drop), b (clear), c (overcast)
S = catch_up(5)
days = S.days
samples_kw = {label: S.samples[label] * dd["capacity_kwp"] for label, dd in days.items()}
d, s = days["b"], samples_kw["b"]
display(Markdown(f"📄 [Metrics at a glance](https://github.com/{c3.REPO}/blob/main/docs/metrics_at_a_glance.pdf)"))

**Ex7.** Point estimate (median), central prediction interval and RMSE. `samples` has shape `[N, T]`; return
one value per half-hour for each output.

In [ ]:
def point_and_band(samples, level=80):
    '''Median and central prediction interval of `level` %.'''
    q = ____
    median, lower, upper = ____
    return median, lower, upper


def rmse(pv, pv_hat):
    return ____


ex7.check(point_and_band, rmse)
# ex7.hint()
# ex7.solution()

In [ ]:
point_and_band, rmse = ex7.use(point_and_band, rmse)
med, lo, up = point_and_band(s, 80)
plots.band(d, med, lo, up, 80)
for label, dd in days.items():
    print(f"day {label}: RMSE of the median {rmse(dd['pv_kw'], point_and_band(samples_kw[label])[0]):.3f} kW (all 48 half-hours)")

**Ex8.** Calibration (PICP, in %) and sharpness (PINAW, in kW/kWp), on daylight half-hours only. `pv`,
`lower` and `upper` are in kW, `capacity` in kWp.

In [ ]:
def picp_pinaw(pv, lower, upper, ghi, capacity):
    '''PICP and PINAW of one day.'''
    sun = ____
    inside = ____
    picp = ____
    pinaw = ____
    return picp, pinaw

ex8.check(picp_pinaw)
# ex8.hint()

In [ ]:
picp_pinaw = ex8.use(picp_pinaw)
for label, dd in days.items():
    for level in (50, 80):
        _, lo, up = point_and_band(samples_kw[label], level)
        p, w = picp_pinaw(dd["pv_kw"], lo, up, dd["ghi"], dd["capacity_kwp"])
        print(f"day {label}, {level} % PI:  PICP {p:5.1f} %   PINAW {w:.3f} kW/kWp")

One day is not enough to judge calibration: by chance a single day can land above or below the promised level
(C3-A7). Over 300 days, the 80 % PI of this model covered 86.5 % of the true values.

**Ex8c (optional). How many samples are enough?** We draw $N$ = 10, 50 and 200 samples, five times each, and look at
how much the 10 %, 50 % and 90 % quantiles at noon and the PICP change between repeats.

In [ ]:
if not GPU:
    print("Ex8c skipped without a GPU.")
else:
    S8 = catch_up(4)
    noon = data.TIME.index("12:00")
    rows = {10: [], 50: [], 200: []}
    for r in range(5):
        s200 = postprocess(model.sample(S8.model, d["c"], S8.schedule, reverse_step, n=200, seed=100 + r), d["ghi"])
        s200 = s200 * d["capacity_kwp"]
        for n in rows:
            med, lo, up = point_and_band(s200[:n], 80)
            p, _ = picp_pinaw(d["pv_kw"], lo, up, d["ghi"], d["capacity_kwp"])
            rows[n].append((lo[noon].item(), med[noon].item(), up[noon].item(), p.item()))
    plots.repeats(rows)

<div style="background:rgba(232,160,32,0.13);border-left:5px solid #e8a020;padding:10px 16px;border-radius:4px">
<b>q6.</b> With 50 samples, the 10 % quantile rests on about 5 samples. How many samples per day would the 1 % quantile need to rest on 5, and how many network passes is that with K = 100?
<ul style="margin:8px 0 2px 0"><li><b>A</b>&nbsp; 100 samples, 10 000 passes</li><li><b>B</b>&nbsp; 500 samples, but only 100 passes, because all samples run in one batch</li><li><b>C</b>&nbsp; 500 samples, 50 000 passes</li><li><b>D</b>&nbsp; about 5 000 samples, because the error of a quantile estimate only shrinks with √N</li></ul>
</div>

In [ ]:
q6.check(____)   # A, B, C or D

**Ex9.** The pinball loss scores all quantiles at once. `q_pred` holds the predicted quantiles, shape
`[Q, T]`, for the levels `q`, shape `[Q]`; average over the levels and the daylight half-hours.

In [ ]:
def pinball(pv, q_pred, q, ghi):
    '''Mean pinball loss of one day.'''
    err = ____
    loss = ____
    return ____

ex9.check(pinball)
# ex9.hint()

In [ ]:
pinball = ex9.use(pinball)
q = torch.arange(1, 10) / 10
for label, dd in days.items():
    pl = pinball(dd["pv_kw"], torch.quantile(samples_kw[label], q, dim=0), q, dd["ghi"])
    print(f"day {label}: pinball loss {pl:.4f} kW = {pl / dd['capacity_kwp']:.4f} kW/kWp")

<div style="height:8px;background:linear-gradient(90deg,#1f5f99,#3a8fd6);border-radius:4px"></div>

## <span style="color:#2f7fc8">Section 6</span> · What does the model listen to? <small>(≈ 20 min)</small>

In C3-A6 (screen 3) you compared the three condition sets on three days. Now we test them against each other.

**Ex8b.** We give the model with all three conditions inputs that contradict each other (all real values for
customer 134):

- **(a)** the overcast day (c) with the GHI of the clear day (b), keeping its real $\mathrm{NC}_d$ and $\mathrm{NC}_{d-1}$;
- **(a′)** the overcast day with the clear day's $\mathrm{NC}_d$, keeping its real GHI;
- **(b)** the clear day with $\mathrm{NC}_d$ drawn as a straight line from 07:00 to 18:00, so the household no longer
  appears to export (the values stay within the observed range).

<div style="background:rgba(232,160,32,0.13);border-left:5px solid #e8a020;padding:10px 16px;border-radius:4px">
<b>q5.</b> <i>Predict first.</i> Will the samples follow the sky (GHI) or the meter (NC)?
<ul style="margin:8px 0 2px 0"><li><b>A</b>&nbsp; the sky: GHI is the physical driver of PV, and the network sees it at every denoising step</li><li><b>B</b>&nbsp; in between: the network combines both inputs, so the samples land between the two cases and the spread grows</li><li><b>C</b>&nbsp; whichever input signals more PV: the sky in (a), the meter in (a′) and (b)</li><li><b>D</b>&nbsp; the meter: in training, NC<sub>d</sub> carried the household-specific PV imprint, so the network learned to trust it more than the area-wide GHI</li></ul>
</div>

In [ ]:
q5.check(____)   # A, B, C or D

In [ ]:
# Catch-up, Section 6
S = catch_up(6)
schedule, models = S.schedule, S.models
reverse_step = ex5.use(globals().get("reverse_step", reference.reverse_step))
postprocess = ex6.use(globals().get("postprocess", reference.postprocess))
point_and_band, rmse = ex7.use(globals().get("point_and_band", reference.point_and_band),
                               globals().get("rmse", reference.rmse))
picp_pinaw = ex8.use(globals().get("picp_pinaw", reference.picp_pinaw))
pinball = ex9.use(globals().get("pinball", reference.pinball))

In [ ]:
clear, dull = (data.day(data.CUSTOMER, data.DAYS_134[label]) for label in "bc")
nc_flat = clear["nc_pu"].clone()
nc_flat[14:37] = torch.linspace(nc_flat[14].item(), nc_flat[36].item(), 23)   # 07:00 to 18:00
cases = {   # name: (conditions, day whose true PV is shown)
    "overcast, real":                (dull["c"], dull),
    "(a) overcast + clear-sky GHI":  (torch.stack([clear["ghi"], dull["nc_pu"], dull["nc_lag1_pu"]]), dull),
    "(a′) overcast + clear NC_d":    (torch.stack([dull["ghi"], clear["nc_pu"], dull["nc_lag1_pu"]]), dull),
    "clear, real":                   (clear["c"], clear),
    "(b) clear, NC_d without export": (torch.stack([clear["ghi"], nc_flat, clear["nc_lag1_pu"]]), clear),
}
raw = model.sample(models["all"], torch.stack([c for c, _ in cases.values()]), schedule, reverse_step)
panels = []
for i, (name, (c, dd)) in enumerate(cases.items()):
    s = postprocess(raw[i], c[0])
    med, lo, up = point_and_band(s * dd["capacity_kwp"], 80)
    print(f"{name:32s} median energy {med.sum() / 2:5.2f} kWh (true {dd['pv_kw'].sum() / 2:.2f})"
          f"   mean width of the 80 % PI {(up - lo)[c[0] > 0].mean():.3f} kW")
    panels.append((s, dd, name))
plots.compare(2, 3, panels)

The samples follow the meter. With the clear-sky GHI, the overcast day's PV stays low and the spread grows (a);
with the clear day's export, the overcast day suddenly gets a lot of PV (a′); and without the midday export, the
clear day loses most of its PV (b).

The model **learned from data, not physics**. In the 2012–13 Ausgrid data a clear day almost always comes with a
midday export, so the model learned "no export ⇒ no PV". On 21 test households the pattern held for most; it was
weak for the few that export little even on clear days (customers 12, 25, 149 and 276), where GHI weighs more.

<div style="background:rgba(232,160,32,0.13);border-left:5px solid #e8a020;padding:10px 16px;border-radius:4px">
<b>q7.</b> In (b) the model is confidently wrong. Which remedy addresses the cause rather than the symptom?
<ul style="margin:8px 0 2px 0"><li><b>A</b>&nbsp; retrain with households whose PV is partly consumed on site (batteries, large daytime loads)</li><li><b>B</b>&nbsp; generate 500 samples instead of 50, so that the interval also covers the true PV</li><li><b>C</b>&nbsp; drop NC<sub>d</sub> from the conditions, so that the meter can no longer mislead the model</li><li><b>D</b>&nbsp; add a longer history (NC<sub>d−2</sub> … NC<sub>d−7</sub>), so that the model sees the household's usual export</li></ul>
</div>

In [ ]:
q7.check(____)   # A, B, C or D

**Compare the three course models** (GHI only, NC only, all three) on several test days, with your functions from Section 5.
MASE is computed by the package (C3-A7: below 1 means better than repeating the previous half-hour).

In [ ]:
days6 = [data.day(cust, date) for cust, date in data.experiment_days()[: 8 if GPU else 3]]
c6 = torch.stack([dd["c"] for dd in days6])
q = torch.arange(1, 10) / 10
rows, samples6 = [], {}
for name, m in models.items():
    raw = model.sample(m, c6, schedule, reverse_step)
    for i, dd in enumerate(days6):
        s = postprocess(raw[i], dd["ghi"]) * dd["capacity_kwp"]
        samples6[name, i] = s
        med, lo, up = point_and_band(s, 80)
        p, w = picp_pinaw(dd["pv_kw"], lo, up, dd["ghi"], dd["capacity_kwp"])
        pl = pinball(dd["pv_kw"], torch.quantile(s, q, dim=0), q, dd["ghi"]) / dd["capacity_kwp"]
        rows.append({"model": name, "day": f"{dd['customer']} {dd['date']}", "RMSE (kW)": float(rmse(dd["pv_kw"], med)),
                     "MASE": float(reference.mase(dd["pv_kw"], med)), "PICP80 (%)": float(p),
                     "PINAW80 (kW/kWp)": float(w), "pinball (kW/kWp)": float(pl)})

import pandas as pd
table = pd.DataFrame(rows)
summary = table.groupby("model", sort=False).agg({"RMSE (kW)": "median", "MASE": "median", "PICP80 (%)": "mean",
                                                  "PINAW80 (kW/kWp)": "mean", "pinball (kW/kWp)": "mean"})
print(f"{len(days6)} test days; RMSE and MASE: median over days, the rest: mean")
display(summary.round(3))
plots.compare(3, 3, [(samples6[n, i] / days6[i]["capacity_kwp"], days6[i], f"{n}, {days6[i]['date']}")
                     for n in models for i in range(3)])

<div style="background:rgba(232,160,32,0.13);border-left:5px solid #e8a020;padding:10px 16px;border-radius:4px">
<b>q8.</b> In the table, the NC-only model has intervals about as narrow as (or narrower than) the model with all three conditions, yet a higher pinball loss. What explains this?
<ul style="margin:8px 0 2px 0"><li><b>A</b>&nbsp; the pinball loss rewards wider intervals, which favours the model with all three conditions</li><li><b>B</b>&nbsp; its quantiles are as tight but more often in the wrong place (e.g. on dull days); the pinball loss penalises misplaced quantiles, PINAW only measures width</li><li><b>C</b>&nbsp; the pinball loss is computed over all 48 half-hours, so the night zeros weigh differently for each model</li><li><b>D</b>&nbsp; PINAW is averaged over the days while the pinball loss is a median, so the two columns are not comparable</li></ul>
</div>

In [ ]:
q8.check(____)   # A, B, C or D

<div style="height:8px;background:linear-gradient(90deg,#1f5f99,#3a8fd6);border-radius:4px"></div>

## <span style="color:#2f7fc8">Section 7</span> · Wrap-up <small>(≈ 5 min)</small>

In [ ]:
progress()
display(Markdown(f"📄 Keep for reference: [Metrics at a glance](https://github.com/{c3.REPO}/blob/main/docs/metrics_at_a_glance.pdf)"))

**Key idea.** A conditional diffusion model learns to predict the noise added to real profiles. Run backwards from
pure noise, and guided at every step by smart-meter and weather data, it generates many plausible PV profiles; their
quantiles give point estimates and prediction intervals, judged by calibration and sharpness. What the model knows
comes from its training data: conditions are evidence, weighed as the data taught it, not physical rules.

**Optional extensions** (GPU, a few minutes each): change the number of steps $K$ or the width $B_{ch}$, retrain
and compare with the course model. Fewer steps make sampling faster; a narrower network has a quarter of the weights
at half the width (C3-A6, screen 4).

In [ ]:
RUN_EXTENSION = False   # set to True to run
K_NEW, WIDTH = 50, 64

if RUN_EXTENSION and GPU:
    S = catch_up(6)
    linear_schedule = ex2.use(globals().get("linear_schedule", reference.linear_schedule))
    diffusion_loss = ex4c.use(globals().get("diffusion_loss", reference.diffusion_loss))
    UNet = ex4b.use(globals().get("UNet1D", model.UNet1D))
    sched_new = linear_schedule(K_NEW)
    net_new = UNet(4, base_channels=WIDTH, time_embed_dim=32, diffusion_steps=K_NEW, depth=2).to(c3.DEVICE)
    plots.losses(model.train(net_new, diffusion_loss, sched_new[2]))
    d = S.day
    plots.compare(1, 2, [(postprocess(model.sample(net_new, d["c"], sched_new, reverse_step), d["ghi"]), d,
                          f"K = {K_NEW}, B_ch = {WIDTH}"),
                         (postprocess(model.sample(S.models["all"], d["c"], S.schedule, reverse_step), d["ghi"]), d,
                          "course model (K = 100, B_ch = 128)")])

### Sources

[1] M. Jené-Vinuesa, H. Kazmi, M. Aragüés-Peñalba and A. Sumper, "Conditional diffusion modeling for probabilistic
behind-the-meter PV disaggregation," *Energy and AI*, art. no. 100692, 2026, doi: 10.1016/j.egyai.2026.100692.
[2] J. Ho, A. Jain and P. Abbeel, "Denoising diffusion probabilistic models," in *Proc. NeurIPS*, vol. 33,
pp. 6840–6851, 2020.
[3] E. L. Ratnam, S. R. Weller, C. M. Kellett and A. T. Murray, "Residential load and rooftop PV generation: an
Australian distribution network dataset," *Int. J. Sustain. Energy*, vol. 36, no. 8, pp. 787–806, 2017.
[4] P. Zippenfenig, "Open-Meteo.com Weather API," Zenodo, 2023, doi: 10.5281/zenodo.7970649.

**Data:** Ausgrid Solar Home Electricity dataset (Creative Commons Attribution) and Open-Meteo (CC BY 4.0); the
preprocessed subset and the course checkpoints are published under CC BY 4.0 (see `DATA_README.md` in the
repository). Code: MIT licence.